In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns

# ── Configure runs ────────────────────────────────────────────────────────────
BASELINE_DIR     = Path("../results/baseline_olympiad_gpt41_n50_20260609")
INTERVENTION_DIR = Path("../results/stage_2_v1_olympiad_gpt41_n50_20260611")

TARGET_FM = "2.6"  # ← swap this to compare any failure mode
# ─────────────────────────────────────────────────────────────────────────────

FM_CODES = [
    "1.1", "1.2", "1.3", "1.4", "1.5",
    "2.1", "2.2", "2.3", "2.4", "2.5", "2.6",
    "3.1", "3.2", "3.3",
]

FM_NAMES = {
    "1.1": "Disobey Task Specification",
    "1.2": "Disobey Role Specification",
    "1.3": "Step Repetition",
    "1.4": "Loss of Conversation History",
    "1.5": "Unaware of Termination Conditions",
    "2.1": "Conversation Reset",
    "2.2": "Fail to Ask for Clarification",
    "2.3": "Task Derailment",
    "2.4": "Information Withholding",
    "2.5": "Ignored Other Agent's Input",
    "2.6": "Action-Reasoning Mismatch",
    "3.1": "Premature Termination",
    "3.2": "No or Incorrect Verification",
    "3.3": "Weak Verification",
}

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

C_BASE = "#2166ac"   # blue  – baseline
C_INT  = "#d6604d"   # red   – intervention

In [ ]:
def load_run(run_dir: Path, label: str) -> pd.DataFrame:
    pred_path = run_dir / "saved_results" / "predictions.csv"
    sum_path  = run_dir / "summary.csv"

    if not pred_path.exists():
        raise FileNotFoundError(
            f"[{label}] Judge predictions not found:\n  {pred_path}\n"
            "Run judge.py on this directory first."
        )

    pred    = pd.read_csv(pred_path)
    correct = pd.read_csv(sum_path)[["trace_id", "correct"]]
    df = pred.merge(correct, on="trace_id", how="left")
    df["_run"] = label
    return df


df_base = load_run(BASELINE_DIR, "Baseline")
df_int  = load_run(INTERVENTION_DIR, "Intervention")

print(f"Baseline      : {len(df_base)} traces  |  pass rate: {df_base['correct'].mean()*100:.1f}%")
print(f"Intervention  : {len(df_int)} traces  |  pass rate: {df_int['correct'].mean()*100:.1f}%")

## 1. Accuracy

In [ ]:
acc_base = df_base["correct"].mean() * 100
acc_int  = df_int["correct"].mean()  * 100
delta_acc = acc_int - acc_base

acc_table = pd.DataFrame({
    "N Traces" : [len(df_base), len(df_int)],
    "Pass Rate": [f"{acc_base:.1f}%", f"{acc_int:.1f}%"],
    "Δ (pp)"   : ["", f"{delta_acc:+.1f}"],
}, index=["Baseline", "Intervention"])

display(acc_table)

fig, ax = plt.subplots(figsize=(4, 3.5))
bars = ax.bar(["Baseline", "Intervention"], [acc_base, acc_int],
              color=[C_BASE, C_INT], width=0.5)
for bar, val in zip(bars, [acc_base, acc_int]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, max(acc_base, acc_int) * 1.25)
ax.set_ylabel("Pass rate (%)")
ax.set_title("Task Accuracy")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
fig.tight_layout()
plt.show()

## 2. Target FM Prevalence

In [ ]:
fm_name   = FM_NAMES[TARGET_FM]
prev_base = df_base[TARGET_FM].mean() * 100
prev_int  = df_int[TARGET_FM].mean()  * 100
delta_fm  = prev_int - prev_base

prev_table = pd.DataFrame({
    "FM"          : [f"{TARGET_FM} — {fm_name}"] * 2,
    "Count"       : [int(df_base[TARGET_FM].sum()), int(df_int[TARGET_FM].sum())],
    "Prevalence"  : [f"{prev_base:.1f}%", f"{prev_int:.1f}%"],
    "Δ (pp)"      : ["", f"{delta_fm:+.1f}"],
}, index=["Baseline", "Intervention"])

display(prev_table)

fig, ax = plt.subplots(figsize=(4, 3.5))
bars = ax.bar(["Baseline", "Intervention"], [prev_base, prev_int],
              color=[C_BASE, C_INT], width=0.5)
for bar, val in zip(bars, [prev_base, prev_int]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10)
max_y = max(prev_base, prev_int)
ax.set_ylim(0, max_y * 1.25 if max_y > 0 else 10)
ax.set_ylabel("% of traces")
ax.set_title(f"FM {TARGET_FM} Prevalence\n{fm_name}")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
fig.tight_layout()
plt.show()

## 3. FM Distribution

In [ ]:
pct_base = (df_base[FM_CODES].sum() / len(df_base) * 100).round(1)
pct_int  = (df_int[FM_CODES].sum()  / len(df_int)  * 100).round(1)
delta_all = (pct_int - pct_base).round(1)

dist_table = pd.DataFrame({
    "Name"           : [FM_NAMES[c] for c in FM_CODES],
    "Baseline %"     : pct_base.values,
    "Intervention %" : pct_int.values,
    "Δ (pp)"         : delta_all.values,
}, index=FM_CODES)
dist_table.index.name = "FM"

dist_table.style.format({
    "Baseline %"     : "{:.1f}%",
    "Intervention %" : "{:.1f}%",
    "Δ (pp)"         : "{:+.1f}",
})

In [ ]:
x = np.arange(len(FM_CODES))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - w / 2, pct_base.values, w, label="Baseline",     color=C_BASE, alpha=0.85)
ax.bar(x + w / 2, pct_int.values,  w, label="Intervention", color=C_INT,  alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(FM_CODES, fontsize=9)
ax.set_ylabel("% of traces")
ax.set_title("Failure Mode Prevalence: Baseline vs Intervention")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

## 4. Summary

In [ ]:
print(f"Accuracy        :  Baseline {acc_base:.1f}%  →  Intervention {acc_int:.1f}%  (Δ = {delta_acc:+.1f} pp)")
print(f"FM {TARGET_FM} prevalence:  Baseline {prev_base:.1f}%  →  Intervention {prev_int:.1f}%  (Δ = {delta_fm:+.1f} pp)")
print()

movers = dist_table[["Name", "Δ (pp)"]].sort_values("Δ (pp)", key=abs, ascending=False).head(5)
print("Largest FM shifts (by |Δ|):")
for fm, row in movers.iterrows():
    sign = "+" if row["Δ (pp)"] >= 0 else ""
    print(f"  {fm}  {row['Name']:<38}  Δ = {sign}{row['Δ (pp)']:.1f} pp")